In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append("../src")

from sentence_transformers import SentenceTransformer
from data_loader import load_candidates
from job_representation import build_job_description
from retrieval import Retriever
from ranking import Ranker

# Load model only once
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

retriever = Retriever(embedding_model)
ranker = Ranker(embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:

from data_loader import load_candidates

candidates = load_candidates(
    "../data/candidates.jsonl",
    limit=100
)

print(len(candidates))


100


In [4]:
from docx import Document
from job_representation import build_job_description

doc = Document("../India_Runs_Hackathon/job_description.docx")

jd_text = "\n".join(
    para.text
    for para in doc.paragraphs
)

job = build_job_description(jd_text)

In [5]:
retriever.build_bm25_index(candidates)

In [6]:
retriever.build_embedding_index(candidates)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
retriever.build_experience_embedding_index(candidates)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Indexed 280 experiences.


In [8]:
retrieved_candidates = retriever.retrieve_candidates(job, top_k=100)

In [10]:
candidate_lookup = {
    c.candidate_id: c
    for c in candidates
}

retrieved_candidates = [
    candidate_lookup[candidate_id]
    for candidate_id, _ in retrieved_candidates
]

In [19]:
retrieved = retriever.retrieve_candidates(job, top_k=100)

candidate_lookup = {
    c.candidate_id: c
    for c in candidates
}

retrieved_candidates = [
    candidate_lookup[cid]
    for cid, _ in retrieved
]

rankings = ranker.rank_candidates(
    job,
    retrieved_candidates
)

Entered _score_evidence
retrieval systems         Weight=3.0 Best=0.361
Entered _score_evidence
ranking systems           Weight=3.0 Best=0.417
Entered _score_evidence
recommendation systems    Weight=3.0 Best=0.508
Entered _score_evidence
search systems            Weight=3.0 Best=0.409
Entered _score_evidence
embeddings                Weight=2.5 Best=0.170
Entered _score_evidence
vector databases          Weight=2.5 Best=0.246
Entered _score_evidence
python                    Weight=2.0 Best=0.200
Entered _score_evidence
llms                      Weight=1.5 Best=0.182
Entered _score_evidence
fine-tuning               Weight=1.0 Best=0.258
Entered _score_evidence
bm25                      Weight=2.0 Best=0.166

Final Evidence Score = 0.184
TOTAL WEIGHT = 23.5
Entered _score_evidence
retrieval systems         Weight=3.0 Best=0.161
Entered _score_evidence
ranking systems           Weight=3.0 Best=0.100
Entered _score_evidence
recommendation systems    Weight=3.0 Best=0.131
Entered _score

In [12]:
# 1
ranker.build_requirement_embeddings(job)

# 2
ranker.build_experience_embeddings(candidates)

In [6]:
results = retriever._experience_embedding_search(
    job.semantic_query,
    top_k=20
)

for rank, (cid, score) in enumerate(results, start=1):

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    print("=" * 100)
    print(f"Rank : {rank}")
    print(f"ID   : {cid}")
    print(f"Score: {score:.3f}")

    for exp in candidate.experiences:
        print(f"\nRole: {exp.title}")
        print(f"Company: {exp.company}")
        print(exp.description[:250])

Rank : 1
ID   : CAND_0000031
Score: 0.557

Role: Recommendation Systems Engineer
Company: Swiggy
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlati

Role: Search Engineer
Company: Mad Street Den
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlati

Role: NLP Engineer
Company: Uber
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlati

Role: Applied ML Engineer
Company: Zomato
Owned the ranking layer fo

In [7]:
results = retriever._experience_embedding_search(
    job.semantic_query,
    top_k=20
)

for cid, score in results:
    print(cid, round(score, 3))

CAND_0000031 0.54
CAND_0000082 0.107
CAND_0000010 0.093
CAND_0000015 0.079
CAND_0000072 0.079
CAND_0000001 0.071
CAND_0000099 0.068
CAND_0000088 0.068
CAND_0000096 0.056
CAND_0000043 0.056
CAND_0000071 0.056
CAND_0000014 0.056
CAND_0000087 0.056
CAND_0000063 0.056
CAND_0000062 0.056
CAND_0000061 0.056
CAND_0000060 0.056
CAND_0000068 0.056
CAND_0000075 0.056
CAND_0000051 0.056


In [8]:
titles = {}

for c in candidates:
    title = c.experiences[0].title
    titles[title] = titles.get(title, 0) + 1

for k, v in sorted(titles.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(v, k)

10 Business Analyst
10 Mechanical Engineer
10 Frontend Engineer
8 Project Manager
7 Graphic Designer
6 Operations Manager
5 Accountant
5 Full Stack Developer
4 Customer Support
4 Marketing Manager
4 QA Engineer
4 HR Manager
4 DevOps Engineer
4 Java Developer
3 Civil Engineer
3 Cloud Engineer
2 Software Engineer
1 Backend Engineer
1 Data Engineer
1 Recommendation Systems Engineer


In [8]:
from collections import Counter

title_counts = Counter()

for candidate in candidates:
    for exp in candidate.experiences:
        title_counts[exp.title] += 1

for title, count in title_counts.most_common(50):
    print(f"{count:4} | {title}")

  28 | Mechanical Engineer
  19 | Business Analyst
  19 | Content Writer
  19 | Project Manager
  18 | Accountant
  16 | HR Manager
  16 | Frontend Engineer
  15 | Marketing Manager
  13 | Operations Manager
  13 | Customer Support
  13 | Graphic Designer
  12 | Civil Engineer
  12 | DevOps Engineer
  10 | Java Developer
  10 | Full Stack Developer
   8 | Cloud Engineer
   7 | Mobile Developer
   6 | QA Engineer
   6 | Software Engineer
   6 | .NET Developer
   4 | Sales Executive
   2 | Backend Engineer
   1 | Analytics Engineer
   1 | Data Engineer
   1 | Recommendation Systems Engineer
   1 | Search Engineer
   1 | NLP Engineer
   1 | Applied ML Engineer
   1 | Data Analyst
   1 | Senior Data Engineer


In [9]:
candidate = ranking[0]["candidate"]

for req, info in candidate.matched_experiences.items():
    print("=" * 50)
    print(req)
    print("Score:", round(info["score"], 3))

    if info["experience"]:
        print("Title:", info["experience"].title)
        print(info["experience"].description[:200])

NameError: name 'ranking' is not defined

In [20]:
print(len(candidates))

100


In [5]:
retriever.build_bm25_index(candidates)
retriever.build_embedding_index(candidates)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
top20 = retriever.retrieve_candidates(
    job,
    top_k=100
    
)

for rank, (cid, score) in enumerate(top20, start=1):

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    print("=" * 100)
    print(f"Rank : {rank}")
    print(f"ID   : {cid}")
    print(f"RRF  : {score:.4f}")

    print(candidate.retrieval_document[:1200])

Rank : 1
ID   : CAND_0000031
RRF  : 0.0328
Company: Swiggy
Experience Title: Recommendation Systems Engineer
Experience Description: Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) — that work was as important as the modeling itself.
Company: Mad Street Den
Experience Title: Search Engineer
Experience Description: Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined w

In [13]:
candidate = candidates[0]

print(candidate.experiences[0])

Experience(company='Mindtree', title='Backend Engineer', duration_months=27, is_current=True, industry='IT Services', company_size='10001+', description='Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure.', evidence_text='Backend Engineer. Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Mo

In [8]:
bm25 = retriever._bm25_search(
    job.document,
    top_k=100
)

print(type(bm25))
print(bm25[0])

<class 'list'>
('CAND_0000031', 438.2318904856046)


In [15]:
bm25_ids = [cid for cid, _ in bm25]

print(
    bm25_ids.index("CAND_0000061")
    if "CAND_0000061" in bm25_ids
    else "Not in Top500"
)

5


In [9]:
embedding = retriever._embedding_search(
    job.semantic_query,
    top_k=20
)

print(type(embedding))
print(embedding[0])

<class 'list'>
('CAND_0000031', 0.4668741524219513)


In [20]:
from sentence_transformers import util

query_emb = retriever.embedding_model.encode(
    job.semantic_query,
    convert_to_tensor=True,
    normalize_embeddings=True
)

for cid in [
    "CAND_0000031",
    "CAND_0000083",
    "CAND_0000074"
]:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    emb = retriever.embedding_model.encode(
        candidate.retrieval_document,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    print(
        cid,
        util.cos_sim(query_emb, emb).item()
    )

CAND_0000031 0.4912135601043701
CAND_0000083 0.25823384523391724
CAND_0000074 0.2561689019203186


In [6]:
#score exp
for cid in [
    "CAND_0000031",
    "CAND_0000082",
    "CAND_0000001",
    "CAND_0000074"
]:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    print(
        cid,
        candidate.metadata["years_of_experience"],
        ranker._score_experience(job, candidate)
    )

CAND_0000031 6.0 1.0
CAND_0000082 7.4 1.0
CAND_0000001 6.9 1.0
CAND_0000074 1.9 0.38


In [7]:
candidate = next(
    c for c in candidates
    if c.candidate_id == "CAND_0000031"
)

print(candidate.signals)

{'profile_completeness_score': 83.4, 'signup_date': '2026-01-28', 'last_active_date': '2026-05-24', 'open_to_work_flag': True, 'profile_views_received_30d': 194, 'applications_submitted_30d': 2, 'recruiter_response_rate': 0.91, 'avg_response_time_hours': 76.1, 'skill_assessment_scores': {'MLflow': 75.1, 'FAISS': 68.4, 'Pinecone': 53.6, 'Image Classification': 57.1}, 'connection_count': 832, 'endorsements_received': 177, 'notice_period_days': 60, 'expected_salary_range_inr_lpa': {'min': 27.3, 'max': 60.2}, 'preferred_work_mode': 'flexible', 'willing_to_relocate': True, 'github_activity_score': 32.6, 'search_appearance_30d': 778, 'saved_by_recruiters_30d': 13, 'interview_completion_rate': 0.6, 'offer_acceptance_rate': 0.38, 'verified_email': False, 'verified_phone': True, 'linkedin_connected': False}


In [8]:
#score signals tets 
for cid in [
    "CAND_0000031",
    "CAND_0000082",
    "CAND_0000001",
    "CAND_0000074"
]:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    print(
        cid,
        round(
            ranker._score_behavior(candidate),
            3
        )
    )

CAND_0000031 0.793
CAND_0000082 0.641
CAND_0000001 0.591
CAND_0000074 0.379


In [13]:
results = ranker.rank_candidates(
    job,
    candidates
)

results[:10]

[{'candidate_id': 'CAND_0000031',
  'final_score': 0.4781632833373547,
  'evidence_score': 0.30928656667470933,
  'skill_score': 0.1,
  'experience_score': 1.0,
  'behavior_score': 0.7926},
 {'candidate_id': 'CAND_0000082',
  'final_score': 0.3926870748651028,
  'evidence_score': 0.18113414973020553,
  'skill_score': 0.16,
  'experience_score': 1.0,
  'behavior_score': 0.6406000000000001},
 {'candidate_id': 'CAND_0000063',
  'final_score': 0.3593620457775891,
  'evidence_score': 0.12292409155517817,
  'skill_score': 0.0,
  'experience_score': 1.0,
  'behavior_score': 0.7395},
 {'candidate_id': 'CAND_0000016',
  'final_score': 0.35622325835585594,
  'evidence_score': 0.13242651671171188,
  'skill_score': 0.0,
  'experience_score': 1.0,
  'behavior_score': 0.7000500000000001},
 {'candidate_id': 'CAND_0000001',
  'final_score': 0.34599858152270313,
  'evidence_score': 0.15569716304540634,
  'skill_score': 0.0,
  'experience_score': 1.0,
  'behavior_score': 0.59075},
 {'candidate_id': 'CAN

In [14]:
results = ranker.rank_candidates(job, candidates)

print(results[:10])

[{'candidate_id': 'CAND_0000031', 'final_score': 0.4781632833373547, 'evidence_score': 0.30928656667470933, 'skill_score': 0.1, 'experience_score': 1.0, 'behavior_score': 0.7926}, {'candidate_id': 'CAND_0000082', 'final_score': 0.3926870748651028, 'evidence_score': 0.18113414973020553, 'skill_score': 0.16, 'experience_score': 1.0, 'behavior_score': 0.6406000000000001}, {'candidate_id': 'CAND_0000063', 'final_score': 0.3593620457775891, 'evidence_score': 0.12292409155517817, 'skill_score': 0.0, 'experience_score': 1.0, 'behavior_score': 0.7395}, {'candidate_id': 'CAND_0000016', 'final_score': 0.35622325835585594, 'evidence_score': 0.13242651671171188, 'skill_score': 0.0, 'experience_score': 1.0, 'behavior_score': 0.7000500000000001}, {'candidate_id': 'CAND_0000001', 'final_score': 0.34599858152270313, 'evidence_score': 0.15569716304540634, 'skill_score': 0.0, 'experience_score': 1.0, 'behavior_score': 0.59075}, {'candidate_id': 'CAND_0000093', 'final_score': 0.3401156597036123, 'evidenc

In [ ]:
import importlib
import ranking

importlib.reload(ranking)

from ranking import Ranker

In [19]:
ranker = Ranker(embedding_model)

In [6]:
#score skill test
candidate = next(
    c for c in candidates
    if c.candidate_id == "CAND_0000031"
)

print(
    ranker._score_skills(
        job,
        candidate
    )
)

0.1


In [7]:
for cid in [
    "CAND_0000031",
    "CAND_0000082",
    "CAND_0000001",
    "CAND_0000074"
]:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    print(
        cid,
        round(
            ranker._score_skills(job, candidate),
            3
        )
    )

CAND_0000031 0.1
CAND_0000082 0.16
CAND_0000001 0.0
CAND_0000074 0.28


In [6]:
#metadata test
candidate = candidates[0]

print(candidate.metadata)

{'num_skills': 17, 'num_jobs': 2, 'num_education': 1, 'years_of_experience': 6.9}


In [ ]:
#score evidence test
andidate = next(
    c for c in candidates
    if c.candidate_id == "CAND_0000031"
)

score = ranker._score_evidence(
    job,
    candidate
)

print(score)

retrieval systems         Weight=3.0 Best=0.000
retrieval systems         Weight=3.0 Best=0.339
retrieval systems         Weight=3.0 Best=0.369
retrieval systems         Weight=3.0 Best=0.369
1.1072203516960144
3.0
ranking systems           Weight=3.0 Best=0.000
ranking systems           Weight=3.0 Best=0.448
ranking systems           Weight=3.0 Best=0.448
ranking systems           Weight=3.0 Best=0.448
2.4500079452991486
6.0
recommendation systems    Weight=3.0 Best=0.000
recommendation systems    Weight=3.0 Best=0.501
recommendation systems    Weight=3.0 Best=0.501
recommendation systems    Weight=3.0 Best=0.501
3.953246980905533
9.0
search systems            Weight=3.0 Best=0.000
search systems            Weight=3.0 Best=0.331
search systems            Weight=3.0 Best=0.417
search systems            Weight=3.0 Best=0.417
5.204311966896057
12.0
embeddings                Weight=2.5 Best=0.000
embeddings                Weight=2.5 Best=0.101
embeddings                Weight=2.5 Best=0.1

In [7]:
for cid in [
    "CAND_0000031",
    "CAND_0000001",
    "CAND_0000047",
    "CAND_0000050"
]:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    score = ranker._score_evidence(job, candidate)

    print(cid, score)

CAND_0000031 0.30928656667470933
CAND_0000001 0.15569716304540634
CAND_0000047 0.10386441245675088
CAND_0000050 0.13281193122267723


In [7]:
test_candidates = [
    "CAND_0000031",
    "CAND_0000074",
    "CAND_0000001",
    "CAND_0000055",
    "CAND_0000082"
]

for cid in test_candidates:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    score = ranker._score_evidence(
        job,
        candidate
    )

    print(f"{cid}: {score:.4f}")

retrieval systems         Weight=3.0 Best=0.000
retrieval systems         Weight=3.0 Best=0.339
retrieval systems         Weight=3.0 Best=0.369
retrieval systems         Weight=3.0 Best=0.369
1.1072203516960144
3.0
ranking systems           Weight=3.0 Best=0.000
ranking systems           Weight=3.0 Best=0.448
ranking systems           Weight=3.0 Best=0.448
ranking systems           Weight=3.0 Best=0.448
2.4500079452991486
6.0
recommendation systems    Weight=3.0 Best=0.000
recommendation systems    Weight=3.0 Best=0.501
recommendation systems    Weight=3.0 Best=0.501
recommendation systems    Weight=3.0 Best=0.501
3.953246980905533
9.0
search systems            Weight=3.0 Best=0.000
search systems            Weight=3.0 Best=0.331
search systems            Weight=3.0 Best=0.417
search systems            Weight=3.0 Best=0.417
5.204311966896057
12.0
embeddings                Weight=2.5 Best=0.000
embeddings                Weight=2.5 Best=0.101
embeddings                Weight=2.5 Best=0.1